# 🚀 Deploy Prático de Small Language Models (SLMs)
**Disciplina:** Inteligência Artificial Generativa (Mestrado)  
**Modelo Base das Demonstrações:** [`Qwen/Qwen2.5-0.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) (~490M parâmetros, arquitetura Transformer com *Grouped-Query Attention* — GQA)

---

## 🎯 Visão Geral da Apresentação
**Small Language Models (SLMs)** permitem levar capacidades generativas para cenários onde LLMs tradicionais (70B+ parâmetros) são inviáveis por custo de GPU, latência de rede ou restrições de privacidade de dados.

Neste notebook centralizado, partimos de **um único modelo base em PyTorch/Safetensors (`BF16`)** e o transformamos passo a passo para **três arquiteturas reais de deploy**, eliminando completamente a dependência do PyTorch em produção:

![Visão Geral da Arquitetura de Deploy de SLMs](assets/fluxo_geral_slms.svg)

| Demo | Cenário de Deploy | Formato do Modelo | Engine de Inferência | Empacotamento / Interface | Tamanho Final |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Demo 1** | **Nuvem / Servidor de Produção** | `.onnx` (*Open Neural Network Exchange*) | **ONNX Runtime** (`CPUExecutionProvider`) | **FastAPI + Uvicorn + Docker Compose** (`:8000`) | **~1890 MB** (`FP32`) |
| **Demo 2** | **Computador / Laptop Local** | `.gguf` (Quantizado em 8-bit `Q8_0`) | **llama.cpp / Ollama** (`mmap` + SIMD) | **Ollama API (`:11435`) + Open WebUI (`:3000`)** | **~506 MB** (`-73%`) |
| **Demo 3** | **Dispositivo Móvel (Android)** | Shards TVM/MLC (`q4f16_1` — 4-bit) + `.so` | **MLC LLM / Apache TVM** (OpenCL GPU) | **APK Nativo Android (`qwen2.5-mlc-chat.apk`)** | **~265 MB** (`-86%`) |

### 📚 Documentações Oficiais de Referência
- **Gerenciador de Pacotes/Ambientes Python:** [Astral `uv` Documentation](https://docs.astral.sh/uv/)
- **Modelo Base:** [Qwen2.5-0.5B-Instruct Model Card (Hugging Face)](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct)
- **Demo 1 (Cloud ONNX):** [Hugging Face Optimum ONNX](https://huggingface.co/docs/optimum/onnxruntime/usage_guides/models) | [Microsoft ONNX Runtime Docs](https://onnxruntime.ai/docs/) | [FastAPI Lifespan Events](https://fastapi.tiangolo.com/advanced/events/)
- **Demo 2 (Edge Local GGUF):** [GGUF Specification (`ggml-org`)](https://github.com/ggml-org/ggml/blob/master/docs/gguf.md) | [llama.cpp Repository](https://github.com/ggml-org/llama.cpp) | [Ollama `Modelfile` Reference](https://github.com/ollama/ollama/blob/main/docs/modelfile.md) | [Open WebUI Docs](https://docs.openwebui.com/)
- **Demo 3 (Mobile Android Nativo):** [MLC LLM Android Deployment Guide](https://llm.mlc.ai/docs/deploy/android.html) | [MLC Model Compilation Pipeline](https://llm.mlc.ai/docs/compilation/compile_models.html) | [Apache TVM Relax Documentation](https://tvm.apache.org/docs/) | [Android `adb` Command Reference](https://developer.android.com/tools/adb)

## 🛠️ Bloco 0: Verificação do Ambiente Local (`uv`, Python e Docker)
Antes de iniciar a **Demo 1**, validamos se o kernel Jupyter está rodando dentro do ambiente virtual isolado (`.venv`) gerenciado pelo [**`uv`**](https://docs.astral.sh/uv/) e se as bibliotecas de exportação (`torch`, `transformers`, `optimum`, `onnxruntime`) estão acessíveis.

> 💡 **Por que o PyTorch está instalado aqui no Notebook, mas NÃO nos containers de produção?**  
> O PyTorch (`torch`) e o `optimum` são usados **apenas na etapa de preparação/exportação** (para ler os pesos originais da Hugging Face e traçar o grafo computacional). Uma vez gerado o arquivo `.onnx`, `.gguf` ou `.tar`, o ambiente de produção roda sem PyTorch!

In [26]:
import sys
import torch
import onnxruntime as ort
import transformers
import optimum

# sys.executable confirma que estamos usando o interpretador Python isolado da pasta .venv criada pelo 'uv'
print(f"Python executável : {sys.executable}")

# Exibe a versão exata do Python (3.12.x) garantindo compatibilidade de ABI com os pacotes C++
print(f"Versão do Python  : {sys.version.split()[0]}")

# Verifica o PyTorch (usado apenas no notebook para exportar o modelo HF -> ONNX)
print(f"PyTorch           : {torch.__version__} (CUDA disponível localmente: {torch.cuda.is_available()})")

# transformers carrega a arquitetura Qwen2 e o AutoTokenizer (tokenização Byte-Level BPE)
print(f"Transformers      : {transformers.__version__}")

# ort.get_available_providers() lista os backends de hardware que o ONNX Runtime encontrou nesta máquina
print(f"ONNX Runtime      : {ort.__version__} | Providers disponíveis: {ort.get_available_providers()}")

Python executável : C:\Users\Pichau\projects\ia_generativa\.venv\Scripts\python.exe
Versão do Python  : 3.12.13
PyTorch           : 2.14.1+cpu (CUDA disponível localmente: False)
Transformers      : 4.57.6
ONNX Runtime      : 1.31.0 | Providers disponíveis: ['AzureExecutionProvider', 'CPUExecutionProvider']


---
# ☁️ Demo 1: Servidor Externo / Produção (FastAPI + ONNX Runtime via Docker)

![Fluxo Arquitetural da Demo 1: Exportação ONNX e Microsserviço FastAPI](assets/fluxo_demo1_onnx.svg)

### Por que usar ONNX Runtime + FastAPI + Docker na nuvem?
1. **Zero PyTorch em Produção:** Após converter o modelo para `.onnx`, o container de produção **não precisa do PyTorch instalado** (economizando ~3.5 GB de imagem Docker). Ele roda exclusivamente com `onnxruntime` + `numpy`.
2. **Otimizações de Grafo ([`GraphOptimizationLevel`](https://onnxruntime.ai/docs/performance/model-optimizations/graph-optimizations.html)):** O ONNX Runtime aplica *Constant Folding*, eliminação de nós redundantes e **fusão de operadores** (ex.: fundir múltiplas operações de *Attention*, *LayerNorm* e *MatMul* em um único kernel C++ otimizado).
3. **Portabilidade de Hardware ([`Execution Providers`](https://onnxruntime.ai/docs/execution-providers/)):** O mesmo artefato `.onnx` pode rodar em CPU (`CPUExecutionProvider`), GPU NVIDIA (`CUDAExecutionProvider` / `TensorRT`), Intel (`OpenVINO`) ou AMD (`ROCm`) mudando apenas 1 linha de configuração.
4. **Microsserviço Isolado:** O container Docker empacota a API REST ([`FastAPI`](https://fastapi.tiangolo.com/)) com documentação interativa automática (`Swagger UI`) pronta para orquestração (Kubernetes, Cloud Run, AWS ECS).

### Passo 1.1 — Exportando o Modelo Hugging Face para ONNX
Utilizamos a classe [**`ORTModelForCausalLM`**](https://huggingface.co/docs/optimum/onnxruntime/package_reference/modeling_ort#optimum.onnxruntime.ORTModelForCausalLM) da biblioteca `optimum.onnxruntime` para converter o modelo `Qwen/Qwen2.5-0.5B-Instruct` do formato PyTorch (`safetensors`) para `.onnx`.

- **O que o parâmetro `export=True` faz por baixo dos panos?**
  1. Instancia o modelo PyTorch original em modo de avaliação (`model.eval()`).
  2. Cria tensores simbólicos (*dummy inputs*) para `input_ids`, `attention_mask`, `position_ids` e para as 24 camadas de **KV-Cache** (`past_key_values.0..23`).
  3. Executa `torch.onnx.export` convertendo cada operação PyTorch (`nn.Linear`, `RMSNorm`, `RotaryEmbedding`, `ScaledDotProductAttention`) em nós padronizados do grafo ONNX.
  4. Como os tensores em `FP32` somam ~1.89 GB (próximo ao limite de 2 GB do formato Protobuf), o ONNX salva a topologia do grafo em `model.onnx` (~1 MB) e os pesos binários contíguos em `model.onnx_data` (~1889 MB).

In [ ]:
import gc
from pathlib import Path
import time
from optimum.onnxruntime import ORTModelForCausalLM
from transformers import AutoTokenizer

# Identificador oficial do repositório na Hugging Face e pasta local de destino
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ONNX_OUTPUT_DIR = Path("./models/qwen2.5-0.5b-onnx")

# Evita re-exportar caso o arquivo model.onnx já tenha sido gerado anteriormente
if (ONNX_OUTPUT_DIR / "model.onnx").exists():
    print(f"✅ Modelo ONNX já exportado em: {ONNX_OUTPUT_DIR.resolve()}")
else:
    print(f"⏳ Baixando '{MODEL_ID}' e convertendo para o formato ONNX...")
    t0 = time.perf_counter()
    
    # export=True: realiza o tracing simbólico do modelo PyTorch e converte para grafo ONNX com KV-Cache ativo
    # provider="CPUExecutionProvider": define que a validação pós-exportação rodará na CPU
    ort_model = ORTModelForCausalLM.from_pretrained(
        MODEL_ID,
        export=True,
        provider="CPUExecutionProvider",
    )
    
    # Baixa o tokenizador (vocabulário Byte-Level BPE de 151.665 tokens + template ChatML)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    
    # Persiste no disco: model.onnx (estrutura do grafo), model.onnx_data (pesos FP32) e tokenizer.json
    ONNX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    ort_model.save_pretrained(ONNX_OUTPUT_DIR)
    tokenizer.save_pretrained(ONNX_OUTPUT_DIR)
    
    # Importante: remove o objeto da memória RAM do Jupyter e força o Garbage Collector
    # para não competir por memória com os containers Docker que subiremos a seguir
    del ort_model
    gc.collect()
    
    elapsed = time.perf_counter() - t0
    print(f"✅ Exportação concluída em {elapsed:.1f}s! Arquivos salvos em: {ONNX_OUTPUT_DIR.resolve()}")

# Lista todos os arquivos gerados mostrando a separação entre topologia (model.onnx) e pesos (model.onnx_data)
print("\n📂 Arquivos gerados no diretório ONNX:")
for f in sorted(ONNX_OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f" - {f.name:<30} ({size_mb:>8.2f} MB)")

### Passo 1.2 — Otimização de Grafo e Inspeção da `InferenceSession` (ONNX Runtime)
Agora que temos o arquivo `model.onnx`, vamos abrir uma sessão nativa do [**`onnxruntime.InferenceSession`**](https://onnxruntime.ai/docs/api/python/api_summary.html#inferencesession) habilitando todas as otimizações de grafo (`ORT_ENABLE_ALL`) e inspecionar sua assinatura de tensores:

1. **Entradas Primárias (3 tensores):**
   - `input_ids` (`[batch_size, sequence_length]`): IDs inteiros (`int64`) dos tokens de entrada.
   - `attention_mask` (`[batch_size, total_sequence_length]`): Máscara indicando quais tokens devem ser considerados na atenção.
   - `position_ids` (`[batch_size, sequence_length]`): Posição absoluta de cada token para cálculo do *Rotary Position Embedding (RoPE)*.
2. **Entradas de KV-Cache (48 tensores = 24 camadas × 2 `[key, value]`):**
   - `past_key_values.0.key`, `past_key_values.0.value`, ..., `past_key_values.23.value`.
   - Formato `[batch_size, 2, past_sequence_length, 64]`: O número `2` reflete o **Grouped-Query Attention (GQA)** do Qwen2.5-0.5B (14 cabeças de *Query* compartilham apenas **2 cabeças de *Key/Value***, reduzindo o consumo de memória do KV-Cache em 7x!).
3. **Saídas (49 tensores):**
   - `logits` (`[batch_size, sequence_length, 151936]`): Pontuações não-normalizadas para cada token do vocabulário.
   - `present.0.key` .. `present.23.value`: O novo estado do KV-Cache atualizado após processar o(s) token(s) atual(is).

In [ ]:
import gc
import onnxruntime as ort

onnx_file_path = ONNX_OUTPUT_DIR / "model.onnx"

# SessionOptions permite configurar threads, profiling e nível de otimização do grafo C++
sess_options = ort.SessionOptions()

# ORT_ENABLE_ALL ativa otimizações Nível 1 (Constant Folding), Nível 2 (Fusão de Operadores Gemm/MatMul+Add+Gelu)
# e Nível 3 (Otimizações de layout de memória NCHWc para SIMD AVX2/AVX-512 na CPU)
sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

# Carrega o grafo e aplica as otimizações em tempo de inicialização
session = ort.InferenceSession(
    str(onnx_file_path),
    sess_options=sess_options,
    providers=["CPUExecutionProvider"],
)

# Extrai metadados simbólicos de todas as conexões de entrada e saída do grafo
inputs = session.get_inputs()
outputs = session.get_outputs()

print(f"🔧 Execution Provider Ativo : {session.get_providers()}")
print(f"📥 Total de Entradas no Grafo: {len(inputs)} (3 primárias + KV-Cache das {(len(inputs)-3)//2} camadas Transformer)")
print("\nPrimeiras 5 entradas do grafo ONNX:")
for inp in inputs[:5]:
    print(f"  • {inp.name:<28} shape={inp.shape}  type={inp.type}")

print(f"\n📤 Total de Saídas no Grafo  : {len(outputs)} (1 tensor de logits + 48 tensores present.X.key/value)")
print("Primeiras 3 saídas do grafo ONNX:")
for out in outputs[:3]:
    print(f"  • {out.name:<28} shape={out.shape}  type={out.type}")

# Libera a sessão local da memória RAM (~2 GB) para deixar o computador leve para o container Docker!
del session
gc.collect()
print("\n🧹 Sessão local liberada da memória RAM (pronto para subir o container Docker).")

### Passo 1.3 — Revisão da Arquitetura da API (`FastAPI` + `Dockerfile` + `docker-compose.yml`)
Para simular o deploy em nuvem/produção, construímos três arquivos principais na pasta [`demo1_cloud_onnx/`](demo1_cloud_onnx/):

1. **[`demo1_cloud_onnx/app/main.py`](demo1_cloud_onnx/app/main.py)**:
   - Usa o gerenciador de contexto [`lifespan`](https://fastapi.tiangolo.com/advanced/events/) do FastAPI para carregar a `onnxruntime.InferenceSession` **uma única vez** quando o servidor sobe (evitando recarregar 1.89 GB a cada requisição HTTP).
   - Implementa a função `generate_with_onnx()` usando **apenas NumPy + ONNX Runtime** (sem PyTorch!):
     - **Passo 0 (*Prefill*):** Processa todos os tokens do prompt de uma só vez (`input_ids` com shape `[1, seq_len]`) e inicializa o `past_key_values` com tensores vazios (`shape=[1, 2, 0, 64]`).
     - **Passos 1..N (*Decoding Autoregressivo*):** A cada iteração, passa **apenas o último token gerado** (`shape=[1, 1]`) junto com o `present.X.key/value` da iteração anterior reaproveitado como `past_key_values.X.key/value`.
     - **Amostragem (*Sampling*):** Aplica escala de `temperature`, estabilização numérica (`logits - max(logits)`), `softmax` e *Nucleus Sampling* (`top_p`) diretamente em NumPy, parando imediatamente ao encontrar os tokens de fim de turno (`<|im_end|>` ou `<|endoftext|>`).
2. **[`demo1_cloud_onnx/Dockerfile`](demo1_cloud_onnx/Dockerfile)**:
   - Imagem enxuta `python:3.12-slim` contendo apenas `onnxruntime`, `fastapi`, `uvicorn`, `numpy` e `transformers` (~40 MB de bibliotecas Python, **zero PyTorch**).
3. **[`demo1_cloud_onnx/docker-compose.yml`](demo1_cloud_onnx/docker-compose.yml)**:
   - Mapeia a porta `8000:8000`, limita recursos (`cpus: 4.0`, `mem_limit: 4g`) e monta `./models/qwen2.5-0.5b-onnx` como volume somente-leitura (`/app/model_onnx:ro`).

### Passo 1.4 — Construindo e Subindo o Container Docker
A célula abaixo executa o comando [`docker compose up -d --build`](https://docs.docker.com/reference/cli/docker/compose/up/) para construir a imagem da API e iniciar o container em segundo plano.

> **Detalhamento dos argumentos do comando:**
> - `-f demo1_cloud_onnx/docker-compose.yml`: Especifica qual arquivo de orquestração usar (mantendo cada Demo isolada em sua própria subpasta).
> - `up -d`: Cria a rede virtual, monta o volume com o `model.onnx` e inicia o container em modo *detached* (sem travar o terminal do Jupyter).
> - `--build`: Recompila a imagem Docker automaticamente caso você tenha alterado o `main.py` ou o `requirements.txt`.

In [ ]:
import subprocess

# Monta o comando declarativo do Docker Compose para a Demo 1
cmd = ["docker", "compose", "-f", "demo1_cloud_onnx/docker-compose.yml", "up", "-d", "--build"]
print("🚀 Executando:", " ".join(cmd))

# encoding="utf-8" e errors="replace" garantem que caracteres especiais da saída do Docker
# não gerem UnicodeDecodeError no console padrão (cp1252) do Windows
result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(result.stdout)
if result.returncode != 0:
    print("⚠️ Saída de erro/aviso:\n", result.stderr)
else:
    print("✅ Container 'slm-onnx-api' iniciado com sucesso!")

In [ ]:
import time
import requests

API_BASE_URL = "http://localhost:8000"

# Polling de Healthcheck: como o ONNX Runtime leva ~2 a 5 segundos aplicando otimizações de grafo (ORT_ENABLE_ALL)
# ao carregar os 1.89 GB do modelo na inicialização do Uvicorn, aguardamos o endpoint GET /health responder 200 OK
print("⏳ Aguardando a API FastAPI carregar o modelo ONNX em memória...")
for attempt in range(1, 31):
    try:
        # Faz uma chamada HTTP leve com timeout curto (2s) para verificar se o servidor já subiu
        resp = requests.get(f"{API_BASE_URL}/health", timeout=2)
        if resp.status_code == 200 and resp.json().get("status") == "ok":
            print(f"✅ API pronta na tentativa {attempt}!")
            print("📊 Resposta do /health:", resp.json())
            break
    except requests.exceptions.RequestException:
        # Enquanto o Uvicorn ainda está inicializando a InferenceSession, a conexão é recusada; aguardamos e tentamos novamente
        pass
    time.sleep(2)
else:
    print("❌ A API não respondeu a tempo. Verifique os logs com a célula abaixo.")

In [ ]:
# Exibe as últimas 25 linhas de log do container Docker
# Mostra o momento exato em que o lifespan do FastAPI carrega o modelo ONNX e abre a porta 8000 no Uvicorn
!docker compose -f demo1_cloud_onnx/docker-compose.yml logs --tail=25

### Passo 1.5 — Testando a API em Produção (`POST /generate` e Swagger UI)
Com o container rodando na porta `8000`, podemos validar o microsserviço de três formas:
1. **Abrir a documentação interativa do Swagger UI (OpenAPI 3.1) no navegador:** 👉 **[http://localhost:8000/docs](http://localhost:8000/docs)**
2. **Consultar os metadados do grafo carregado via `GET /model-info`** (confirmando a quantidade de camadas, cabeças KV e dimensão de cada cabeça).
3. **Disparar uma requisição `POST /generate`** diretamente daqui do Notebook e medir a latência total (`ms`) e a vazão (`tokens/s`) em CPU.

In [ ]:
import json
import requests

# Consulta o endpoint GET /model-info que inspeciona dinamicamente a InferenceSession dentro do container
info_resp = requests.get(f"{API_BASE_URL}/model-info", timeout=10)
print("🔍 Metadados do Grafo ONNX retornados pelo container (GET /model-info):")
print(json.dumps(info_resp.json(), indent=2, ensure_ascii=False))

In [ ]:
# Define o payload JSON validado pelo modelo Pydantic (InputData) lá no FastAPI
payload = {
    "prompt": "Explique em 3 tópicos curtos por que Small Language Models (SLMs) são ideais para deploy em borda e microsserviços.",
    "system_prompt": "Você é um professor de IA Generativa. Responda de forma didática, direta e em português.",
    "max_new_tokens": 256,  # Limite máximo de tokens gerados no loop autorregressivo (suporta até 2048)
    "temperature": 0.2,     # Temperatura baixa deixa a resposta mais determinística e técnica
    "top_p": 0.9            # Nucleus sampling: considera apenas o núcleo de tokens que soma 90% da probabilidade
}

print(f"📤 Enviando requisição POST para {API_BASE_URL}/generate ...\n")
# Envia a requisição HTTP POST para o container Docker
response = requests.post(f"{API_BASE_URL}/generate", json=payload, timeout=120)
data = response.json()

# Exibe o texto decodificado e as métricas de telemetria calculadas pelo servidor
print("🤖 Resposta do Modelo (ONNX Runtime no Docker):")
print("=" * 70)
print(data["generated_text"])
print("=" * 70)
print(f"⚡ Provedor ONNX : {data['execution_provider']}")
print(f"🔢 Tokens Prompt : {data['prompt_tokens']} | Novos Tokens: {data['new_tokens']}")
print(f"⏱️ Latência Total: {data['inference_time_ms']} ms")
print(f"🚀 Velocidade    : {data['tokens_per_second']} tokens/s")

### Passo 1.6 — Encerrando o Container da Demo 1 (Opcional)
Após demonstrar o Swagger UI (`http://localhost:8000/docs`) e as chamadas acima, você pode parar o container com `docker compose down` para liberar memória RAM antes da próxima demonstração.

In [ ]:
# Descomente a linha abaixo quando quiser encerrar o container Docker da Demo 1 e liberar ~2 GB de RAM:
# !docker compose -f demo1_cloud_onnx/docker-compose.yml down

---
# 💻 Demo 2: Notebook / Laptop Local (Conversão para GGUF + Ollama + Open WebUI via Docker)

![Fluxo Arquitetural da Demo 2: Conversão GGUF, Modelfile, Ollama e Open WebUI](assets/fluxo_demo2_gguf.svg)

### Por que usar o formato GGUF + `llama.cpp` / Ollama em computadores locais?
1. **Formato de Arquivo Único ([`.gguf` — *GGML Universal File*](https://github.com/ggml-org/ggml/blob/master/docs/gguf.md)):** Criado por Georgi Gerganov para o [**`llama.cpp`**](https://github.com/ggml-org/llama.cpp), o GGUF empacota metadados da arquitetura, vocabulário do tokenizador, template de chat e todos os tensores de pesos em **um único arquivo binário autossuficiente**.
2. **Carregamento Instantâneo via `mmap` (*Memory-Mapped File*):** Os tensores dentro do `.gguf` são alinhados em blocos de memória que o sistema operacional mapeia diretamente do SSD para o espaço de endereçamento do processo (`mmap`), iniciando o modelo em milissegundos sem precisar copiar gigabytes entre buffers.
3. **Quantização por Blocos (`Q8_0`, `Q4_K_M`):** Em vez de armazenar cada peso em `float32` (32 bits) ou `bfloat16` (16 bits), dividimos os pesos em blocos (ex.: blocos de 32 valores) e armazenamos cada peso como inteiro de 8 bits (`int8`) acompanhado de 1 fator de escala `float16` por bloco — reduzindo o tamanho em **~50% sobre o BF16 e ~73% sobre o ONNX FP32** sem perda perceptível de perplexidade!
4. **100% Portátil em Containers:** Subimos tanto o motor [**Ollama**](https://ollama.com/) (`slm-ollama`) quanto a interface gráfica estilo ChatGPT [**Open WebUI**](https://docs.openwebui.com/) (`slm-open-webui`) via `docker-compose.yml` — garantindo que funcione em **qualquer notebook** sem precisar instalar o Ollama no Windows e sem conflitar caso a máquina já tenha outro Ollama instalado!

### Passo 2.1 — Preparando os Pesos Hugging Face (`safetensors`) e o Conversor do `llama.cpp`
Para demonstrar o fluxo real de engenharia (partindo de um modelo aberto da Hugging Face até o arquivo `.gguf`):
1. Garantimos uma cópia local dos arquivos originais do `Qwen/Qwen2.5-0.5B-Instruct` (`model.safetensors`, `config.json`, `tokenizer.json`) em `./models/qwen2.5-0.5b-hf` usando [`snapshot_download`](https://huggingface.co/docs/huggingface_hub/main/en/package_reference/file_download#huggingface_hub.snapshot_download).
2. Baixamos apenas os scripts de conversão Python (`convert_hf_to_gguf.py` e pacote `gguf-py`) do repositório oficial [`ggml-org/llama.cpp`](https://github.com/ggml-org/llama.cpp) usando **Git Sparse-Checkout** (`--filter=blob:none --sparse`), o que baixa menos de 5 MB em segundos em vez de clonar gigabytes de histórico C++.

In [ ]:
from pathlib import Path
import subprocess
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
HF_MODEL_DIR = Path("./models/qwen2.5-0.5b-hf")
LLAMA_CPP_DIR = Path("./demo2_local_gguf/llama.cpp")
CONVERT_SCRIPT = LLAMA_CPP_DIR / "convert_hf_to_gguf.py"

# 1. Garante que o modelo original Hugging Face (.safetensors em BF16) esteja em ./models/qwen2.5-0.5b-hf
if not (HF_MODEL_DIR / "model.safetensors").exists():
    print(f"⏳ Sincronizando arquivos originais do '{MODEL_ID}' para {HF_MODEL_DIR}...")
    # snapshot_download aproveita o cache local do Hugging Face se já tiver sido baixado na Demo 1
    snapshot_download(repo_id=MODEL_ID, local_dir=str(HF_MODEL_DIR))
print(f"✅ Modelo Hugging Face pronto em: {HF_MODEL_DIR.resolve()}")

# 2. Faz clone esparso (sparse-checkout) apenas do conversor oficial convert_hf_to_gguf.py do llama.cpp
if not CONVERT_SCRIPT.exists():
    print("⏳ Clonando scripts de conversão do repositório oficial llama.cpp (sparse-checkout)...")
    # --depth 1 --filter=blob:none: baixa apenas a árvore do commit mais recente sem objetos binários desnecessários
    subprocess.run([
        "git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
        "https://github.com/ggml-org/llama.cpp.git", str(LLAMA_CPP_DIR)
    ], check=True)
    # Seleciona exclusivamente o script convert_hf_to_gguf.py e a biblioteca Python gguf-py
    subprocess.run([
        "git", "-C", str(LLAMA_CPP_DIR), "sparse-checkout", "set", "--no-cone",
        "/convert_hf_to_gguf.py", "/conversion/", "/gguf-py/"
    ], check=True)
print(f"✅ Script conversor do llama.cpp pronto em: {CONVERT_SCRIPT.resolve()}")

### Passo 2.2 — Convertendo e Quantizando de Hugging Face (`safetensors`) para `.gguf`
Agora executamos o [`convert_hf_to_gguf.py`](https://github.com/ggml-org/llama.cpp/blob/master/convert_hf_to_gguf.py) passando:
- **Diretório de origem:** `./models/qwen2.5-0.5b-hf` (contendo `config.json`, `tokenizer.json` e `model.safetensors`).
- **`--outfile`:** `./models/qwen2.5-0.5b-gguf/qwen2.5-0.5b-instruct.gguf`
- **`--outtype q8_0`:** Quantiza os tensores lineares diretamente em blocos de inteiros de 8 bits (`Q8_0`), mantendo embeddings/normas em precisão adequada e gravando toda a especificação GGUF v3 no cabeçalho do arquivo.

In [ ]:
import sys
import time

GGUF_DIR = Path("./models/qwen2.5-0.5b-gguf")
GGUF_DIR.mkdir(parents=True, exist_ok=True)
GGUF_FILE = GGUF_DIR / "qwen2.5-0.5b-instruct.gguf"

# Verifica se a conversão já foi feita anteriormente para economizar tempo durante a apresentação
if GGUF_FILE.exists():
    print(f"✅ Arquivo GGUF já gerado em: {GGUF_FILE.resolve()}")
else:
    # Monta o comando invocando o mesmo interpretador Python do ambiente virtual (sys.executable)
    cmd = [
        sys.executable,
        str(CONVERT_SCRIPT),
        str(HF_MODEL_DIR),
        "--outfile", str(GGUF_FILE),
        "--outtype", "q8_0"  # Opções suportadas pelo script: f32, f16, bf16, q8_0, auto
    ]
    print("🚀 Executando conversão HF -> GGUF (Q8_0):")
    print(" ", " ".join(cmd), "\n")
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
    if proc.returncode != 0:
        print("❌ Erro na conversão:\n", proc.stderr)
    else:
        elapsed = time.perf_counter() - t0
        print(f"✅ Conversão para GGUF concluída em {elapsed:.1f}s!")

# Comparativo didático de tamanho em disco entre os formatos (somando model.onnx + model.onnx_data na Demo 1)
onnx_dir = Path("./models/qwen2.5-0.5b-onnx")
onnx_size = sum(f.stat().st_size for f in onnx_dir.glob("*.onnx*")) / (1024**2) if onnx_dir.exists() else 0
hf_size = (HF_MODEL_DIR / "model.safetensors").stat().st_size / (1024**2)
gguf_size = GGUF_FILE.stat().st_size / (1024**2)

print("\n📊 Comparativo de Tamanho dos Artefatos do Modelo até aqui:")
if onnx_size:
    print(f"  • ONNX (FP32 - Demo 1)        : {onnx_size:>7.1f} MB")
print(f"  • Hugging Face (BF16 Original): {hf_size:>7.1f} MB")
print(f"  • GGUF Quantizado (Q8_0)      : {gguf_size:>7.1f} MB (Redução de {(1 - gguf_size/onnx_size)*100:.1f}% vs ONNX FP32!)")

### Passo 2.3 — Subindo Ollama + Open WebUI via Docker e Importando o `Modelfile`
Assim como o `Dockerfile` descreve uma imagem de container, o [**`Modelfile`**](https://github.com/ollama/ollama/blob/main/docs/modelfile.md) descreve como o Ollama deve empacotar e executar o nosso arquivo `.gguf` ([`demo2_local_gguf/Modelfile`](demo2_local_gguf/Modelfile)):
- `FROM /models_in/qwen2.5-0.5b-instruct.gguf`: Aponta para o arquivo binário `.gguf` montado dentro do container.
- `PARAMETER num_ctx 16384`: Janela de contexto ampla (16K tokens) — **essencial porque interfaces como o Open WebUI enviam histórico de conversa e prompts de sistema/título junto com a pergunta**.
- `PARAMETER num_predict 2048`: Permite respostas longas e detalhadas sem cortar o texto na metade.
- `SYSTEM` e `TEMPLATE`: Define a instrução de sistema em português e a formatação de tokens especiais ChatML (`<|im_start|>` / `<|im_end|>`).

#### 🐳 Arquitetura Portável no [`demo2_local_gguf/docker-compose.yml`](demo2_local_gguf/docker-compose.yml):
1. **`slm-ollama`**: Exposto na porta **`11435`** do host (`11435:11434`) para **nunca dar conflito de porta** caso o computador já tenha um Ollama local rodando na porta `11434`.
2. **`slm-open-webui`**: Interface web exposta em **[http://localhost:3000](http://localhost:3000)**, conectada diretamente ao container `slm-ollama` pela rede interna do Docker (`OLLAMA_BASE_URL=http://ollama:11434`) com login desativado (`WEBUI_AUTH=false`) para facilitar a demonstração.

In [ ]:
import shutil
import subprocess
import time
import requests

CUSTOM_MODEL_NAME = "qwen25-slm-custom"
OLLAMA_DOCKER_URL = "http://localhost:11435"

# 1. Copia o Modelfile para a pasta ./models/qwen2.5-0.5b-gguf (que é montada em /models_in dentro do container slm-ollama)
target_modelfile = GGUF_DIR / "Modelfile"
shutil.copyfile("demo2_local_gguf/Modelfile", target_modelfile)
print("📄 Conteúdo do Modelfile que será importado no Ollama:")
print("-" * 60)
print(target_modelfile.read_text(encoding="utf-8"))
print("-" * 60)

# 2. Sobe os containers 'slm-ollama' e 'slm-open-webui' em background (-d) via Docker Compose
print("🐳 Subindo containers 'slm-ollama' (porta 11435) e 'slm-open-webui' (porta 3000)...")
subprocess.run(
    ["docker", "compose", "-f", "demo2_local_gguf/docker-compose.yml", "up", "-d"],
    check=True,
    encoding="utf-8",
    errors="replace",
)

# 3. Polling: aguarda o daemon do Ollama dentro do container responder na porta 11435
print("⏳ Aguardando servidor 'slm-ollama' iniciar...")
for _ in range(30):
    try:
        # GET /api/tags lista os modelos instalados e confirma que a API HTTP do Ollama está ativa
        if requests.get(f"{OLLAMA_DOCKER_URL}/api/tags", timeout=2).status_code == 200:
            print("✅ Container 'slm-ollama' pronto!")
            break
    except requests.exceptions.RequestException:
        pass
    time.sleep(2)

# 4. Executa 'ollama create' DENTRO do container slm-ollama para registrar o nosso arquivo .gguf + Modelfile
# Nota: em /models_in está montada a pasta ./models/qwen2.5-0.5b-gguf do host
create_cmd = ["docker", "exec", "slm-ollama", "ollama", "create", CUSTOM_MODEL_NAME, "-f", "/models_in/Modelfile"]
print(f"\n⚙️ Importando modelo GGUF no container Ollama: {' '.join(create_cmd)}")
res = subprocess.run(create_cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
print((res.stdout or res.stderr).strip())

print(f"\n🎉 Modelo '{CUSTOM_MODEL_NAME}' registrado no Ollama e Open WebUI ativo em http://localhost:3000 !")

### Passo 2.4 — Testando a Inferência via API do Container Ollama + Chat no Open WebUI
Agora podemos demonstrar o uso do modelo `.gguf` de duas formas complementares:
1. **Via API REST no Notebook:** Fazendo uma chamada `POST http://localhost:11435/api/chat` para o container `slm-ollama` e extraindo as métricas nativas do `llama.cpp` (`eval_count` e `eval_duration`).
2. **Via Interface Gráfica de Chat (Open WebUI):** Abrindo 👉 **[http://localhost:3000](http://localhost:3000)** no navegador, onde o modelo **`qwen25-slm-custom`** já aparece selecionado para conversar com streaming em tempo real!

In [ ]:
import requests

# Payload compatível com a API /api/chat do Ollama (https://github.com/ollama/ollama/blob/main/docs/api.md#generate-a-chat-completion)
chat_payload = {
    "model": CUSTOM_MODEL_NAME,
    "messages": [
        {
            "role": "user",
            "content": "Quais são as principais vantagens do formato GGUF e da quantização para rodar modelos localmente em notebooks?"
        }
    ],
    "stream": False  # stream=False retorna a resposta consolidada em um único JSON junto com as estatísticas de tempo
}

print(f"📤 Enviando pergunta para o modelo '{CUSTOM_MODEL_NAME}' via API do container Ollama ({OLLAMA_DOCKER_URL}/api/chat)...\n")
resp = requests.post(f"{OLLAMA_DOCKER_URL}/api/chat", json=chat_payload, timeout=120)
result = resp.json()

# Extrai o texto gerado e converte os tempos reportados pelo llama.cpp (em nanossegundos) para ms e tokens/s
reply_content = result["message"]["content"]
eval_count = result.get("eval_count", 0)                 # Quantidade de tokens gerados na resposta
eval_duration_ns = result.get("eval_duration", 1)        # Tempo gasto na geração (em nanossegundos, 10^-9 s)
tokens_per_sec = eval_count / (eval_duration_ns / 1e9)   # Cálculo exato de throughput de decodificação
total_duration_ms = result.get("total_duration", 0) / 1e6

print(f"🤖 Resposta do '{CUSTOM_MODEL_NAME}' (GGUF Q8_0 no Container Ollama):")
print("=" * 70)
print(reply_content)
print("=" * 70)
print(f"🔢 Tokens Gerados : {eval_count}")
print(f"⏱️ Tempo Total    : {total_duration_ms:.1f} ms")
print(f"🚀 Velocidade     : {tokens_per_sec:.2f} tokens/s")
print("\n💬 Abra a interface visual de Chat (Open WebUI) em: http://localhost:3000")

### Passo 2.5 — Encerrando os Containers da Demo 2 (Opcional)
Ao concluir a demonstração no Open WebUI (`http://localhost:3000`), você pode encerrar os containers com a célula abaixo.

In [ ]:
# Descomente a linha abaixo quando quiser encerrar o Ollama + Open WebUI:
# !docker compose -f demo2_local_gguf/docker-compose.yml down

---
# 📱 Demo 3: Dispositivo Android Nativo (MLC LLM + Geração de APK)

![Fluxo Arquitetural da Demo 3: Compilação MLC LLM / Apache TVM, Build APK com Gradle e Deploy Offline via ADB](assets/fluxo_demo3_android_mlc.svg)

Chegamos à fronteira mais restrita do deploy de *Small Language Models*: **execução 100% local e offline dentro de um smartphone Android**.

### Por que compilar com [MLC LLM](https://llm.mlc.ai/docs/deploy/android.html) (*Machine Learning Compilation* / [Apache TVM](https://tvm.apache.org/docs/))?
Diferente de um servidor Cloud (Demo 1) ou de um notebook com 16 GB+ de RAM (Demo 2), um celular possui:
1. **Memória RAM/VRAM unificada e extremamente restrita:** O sistema operacional Android possui o *Low Memory Killer (LMK)*, que encerra abruptamente qualquer aplicativo que aloque buffers temporários gigantescos.
2. **GPU Mobile heterogênea (Qualcomm Adreno / ARM Mali):** Não existe NVIDIA CUDA em celulares! Para atingir geração fluida na GPU do smartphone, precisamos traduzir as camadas do Transformer em *kernels* **OpenCL / Vulkan** nativos e aplicar fusão de operadores (*operator fusion*) via **Apache TVM Relax**.
3. **Quantização por Grupo de 4 bits (`q4f16_1`):** Agrupamos blocos de 32 pesos consecutivos, quantizamos cada peso em **4 bits (`int4`)**, empacotamos 8 pesos de 4 bits dentro de um único inteiro de 32 bits (`uint32`) e armazenamos 1 fator de escala `float16` por grupo. Durante a inferência na GPU mobile, o kernel OpenCL faz a desquantização *on-the-fly* diretamente nos registradores da GPU!

### Arquitetura em 4 Etapas desta Demo
1. **Passo 3.1 (Pesos `q4f16_1` & `mlc-chat-config.json`):** Geração da configuração de execução mobile (`context_window_size=4096`, `prefill_chunk_size=512`) e preparação dos shards quantizados em 4 bits, reduzindo o `Qwen2.5-0.5B-Instruct` de **1890 MB** (ONNX FP32) para apenas **~265 MB** (`-86%`)!
2. **Passo 3.2 (Compilação TVM para Android ARM64 + OpenCL):** Execução do compilador [`mlc_llm compile --device android`](https://llm.mlc.ai/docs/compilation/compile_models.html) dentro do container leve [`slm-mlc-compiler`](demo3_android_mlc/Dockerfile), gerando o código nativo (`qwen2.5-0.5b-q4f16_1-android.tar`) e reduzindo o consumo total de memória no celular de **5365 MB para apenas 628.87 MB**!
3. **Passo 3.3 (Build do `.apk` via Gradle):** Compilação do projeto Android nativo ([`MLCChat`](demo3_android_mlc/mlc-llm-src/android/MLCChat) em Kotlin/Jetpack Compose + módulo JNI `:mlc4j` com `libtvm4j_runtime_packed.so`) gerando o instalador **`demo3_android_mlc/qwen2.5-mlc-chat.apk`**.
4. **Passo 3.4 (Instalação + Provisionamento 100% Offline via `adb`):** Instalação do `.apk` direto no celular conectado via USB e transferência dos pesos para o armazenamento interno privado do app (`application.filesDir` via `run-as ai.mlc.mlcchat`), contornando as restrições de *Scoped Storage* do Android 14/15 e permitindo demonstrar o chat no telefone **em Modo Avião (100% Offline)**!

### Passo 3.1 — Quantização `q4f16_1`, `mlc-chat-config.json` e Comparativo dos 4 Formatos

Na célula abaixo:
1. Garantimos que os shards quantizados em 4 bits (`params_shard_0.bin` .. `params_shard_7.bin` + `tensor-cache.json`) estejam presentes em `./models/qwen2.5-0.5b-mlc`.
2. Executamos o comando [`mlc_llm gen_config`](https://llm.mlc.ai/docs/compilation/configure_quantization.html) via container Docker ([`demo3_android_mlc/docker-compose.yml`](demo3_android_mlc/docker-compose.yml)).
   - **Por que passamos `--context-window-size 4096` e `--prefill-chunk-size 512`?**  
     Por padrão, o `config.json` do Qwen2.5 declara `max_position_embeddings = 32768` e `prefill_chunk_size = 8192`. Se compilarmos com esses valores padrão de servidor, o TVM pré-aloca um buffer temporário de **5.05 GB** na RAM do celular! Ao limitar a janela mobile para `4096` e o chunk de prefill para `512`, esse buffer cai para **315 MB**, garantindo estabilidade total em qualquer smartphone.
3. Comparamos lado a lado o tamanho em disco do modelo nas **4 representações** estudadas na apresentação.

In [ ]:
import json
import subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

MLC_MODEL_DIR = Path("./models/qwen2.5-0.5b-mlc")
MLC_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Garante que os shards quantizados em 4 bits (q4f16_1) estejam presentes localmente
# O modelo é dividido em 8 arquivos binários (params_shard_0.bin .. params_shard_7.bin) + tensor-cache.json
shards = list(MLC_MODEL_DIR.glob("params_shard_*.bin"))
if len(shards) < 8:
    print("📥 Baixando shards quantizados q4f16_1 (mlc-ai/Qwen2.5-0.5B-Instruct-q4f16_1-MLC)...")
    snapshot_download(
        repo_id="mlc-ai/Qwen2.5-0.5B-Instruct-q4f16_1-MLC",
        local_dir=str(MLC_MODEL_DIR),
    )
else:
    print(f"✅ {len(shards)} shards quantizados (q4f16_1) já presentes em: {MLC_MODEL_DIR}")

# 2. Gera o mlc-chat-config.json otimizado para memória mobile via container Docker (slm-mlc-compiler)
print("\n⚙️ Gerando mlc-chat-config.json (context_window=4096, prefill_chunk=512) via Docker...")
gen_cmd = [
    "docker", "compose", "-f", "demo3_android_mlc/docker-compose.yml",
    "run", "--rm", "mlc-compiler",          # --rm remove o container efêmero assim que o comando termina
    "mlc_llm", "gen_config",
    "/workspace/model_hf",                  # Volume montado apontando para ./models/qwen2.5-0.5b-hf
    "--quantization", "q4f16_1",            # Quantização por grupo: pesos int4 empacotados em uint32 + escala float16
    "--conv-template", "qwen2",             # Template de conversação ChatML nativo da família Qwen2/2.5
    "--context-window-size", "4096",        # Limita o KV-Cache máximo a 4K tokens (apenas ~48 MB de RAM no celular)
    "--prefill-chunk-size", "512",          # Processa prompts longos em blocos de 512 tokens (reduz buffer temporário em 93%!)
    "-o", "/workspace/dist"                 # Volume montado apontando para ./models/qwen2.5-0.5b-mlc
]
res = subprocess.run(gen_cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")
print(res.stdout or res.stderr)

# 3. Comparativo real de tamanho em disco entre todas as Demos da apresentação
def dir_size_mb(path: Path, pattern: str = "*") -> float:
    """Soma o tamanho (em MB) de todos os arquivos que casam com o padrão glob informado."""
    return sum(f.stat().st_size for f in path.glob(pattern) if f.is_file()) / (1024 ** 2)

onnx_mb = dir_size_mb(Path("./models/qwen2.5-0.5b-onnx"), "*.onnx*")  # Soma model.onnx + model.onnx_data
hf_mb   = dir_size_mb(Path("./models/qwen2.5-0.5b-hf"), "*.safetensors")
gguf_mb = dir_size_mb(Path("./models/qwen2.5-0.5b-gguf"), "*.gguf")
mlc_mb  = dir_size_mb(MLC_MODEL_DIR, "params_shard_*.bin")

print("📊 COMPARATIVO DE TAMANHO DO MODELO (Qwen2.5-0.5B-Instruct):")
print("-" * 68)
print(f"{'Formato / Demo':<36} | {'Precisão':<12} | {'Tamanho (MB)':>12}")
print("-" * 68)
print(f"{'Demo 1: ONNX Runtime (Cloud/CPU)':<36} | {'FP32 (32b)':<12} | {onnx_mb:>9.1f} MB")
print(f"{'Original: Hugging Face Safetensors':<36} | {'BF16 (16b)':<12} | {hf_mb:>9.1f} MB")
print(f"{'Demo 2: GGUF / Ollama (Notebook)':<36} | {'Q8_0 (8b)':<12} | {gguf_mb:>9.1f} MB")
print(f"{'Demo 3: MLC LLM (Android Mobile)':<36} | {'q4f16_1 (4b)':<12} | {mlc_mb:>9.1f} MB")
print("-" * 68)
print(f"💡 Redução total do ONNX FP32 para o MLC 4-bit Mobile: {(1 - mlc_mb/onnx_mb)*100:.1f}% menor!")

### Passo 3.2 — Compilação Nativa do Grafo TVM para Android (`aarch64-linux-android` + OpenCL)

Agora usamos o container `slm-mlc-compiler` para executar o comando [`mlc_llm compile`](https://llm.mlc.ai/docs/compilation/compile_models.html) com `--device android`.

#### 🔬 O que acontece dentro do compilador Apache TVM durante esta célula?
1. **Exportação para TVM Relax:** Constrói a representação intermediária de alto nível (*Relax IR*) do modelo `QWen2Config` com as regras de quantização `GroupQuantize(name='q4f16_1', group_size=32, quantize_dtype='int4', storage_dtype='uint32', model_dtype='float16')`.
2. **Fusão de Operadores (*TIR / Dlight Low-Level Optimizations*):** Funde a desquantização de 4 bits com a multiplicação de matrizes (`fused_dequantize_NT_matmul`) e combina somas residuais com normalização (`fuse_add_norm_prefill`), evitando leituras/escritas intermediárias na memória RAM do celular.
3. **Geração de Código Híbrido (`--target`):**
   - **Host CPU:** `mtriple: aarch64-linux-android` (instruções nativas ARM64 + SIMD ASIMD/NEON compiladas em `lib0.o`).
   - **Device GPU:** `kind: opencl` (kernels OpenCL para GPUs Adreno/Mali empacotados junto com o bytecode da VM Relax em `qwen2_q4f16_1_devc.o`).
4. **Estimativa Estática de Memória:** Calcula exatamente quanto cada função (`prefill`, `decode`, `embed`) vai alocar no celular, comprovando o consumo total de **628.87 MB** com KV-Cache de 4K!

In [ ]:
import subprocess
import tarfile
from pathlib import Path

ANDROID_TAR = Path("./models/qwen2.5-0.5b-mlc/qwen2.5-0.5b-q4f16_1-android.tar")

print("🔨 Compilando grafo TVM Relax + Kernels OpenCL/ARM64 para Android...")
compile_cmd = [
    "docker", "compose", "-f", "demo3_android_mlc/docker-compose.yml",
    "run", "--rm", "mlc-compiler",
    "mlc_llm", "compile",
    "/workspace/dist/mlc-chat-config.json",                 # Lê a configuração com context_window=4096 e prefill_chunk=512
    "--device", "android",                                  # Seleciona automaticamente target OpenCL GPU + host aarch64-linux-android
    "-o", "/workspace/dist/qwen2.5-0.5b-q4f16_1-android.tar" # Exporta o arquivo .tar contendo os objetos nativos (.o)
]

proc = subprocess.run(compile_cmd, capture_output=True, text=True, encoding="utf-8", errors="replace")

# Filtra e exibe as linhas principais do log do compilador TVM (argumentos, target OpenCL e relatório de memória)
output_lines = (proc.stdout + "\n" + proc.stderr).splitlines()
for line in output_lines:
    if any(k in line for k in ["Compiling with", "--target", "--quantization", "[Memory usage]", "Total memory", "Generated:"]):
        print(line)

# Inspeciona o conteúdo interno do arquivo .tar gerado sem precisar extraí-lo no disco
if ANDROID_TAR.exists():
    with tarfile.open(ANDROID_TAR, "r") as tar:
        members = [(m.name, f"{m.size / 1024:.1f} KB") for m in tar.getmembers()]
    print(f"\n✅ Biblioteca Android gerada com sucesso: {ANDROID_TAR} ({ANDROID_TAR.stat().st_size / 1024:.1f} KB)")
    print(f"📦 Objetos nativos ARM64 dentro do pacote: {members}")

### Passo 3.3 — Gerando o Aplicativo Android (`.apk`) com Gradle

Com a biblioteca nativa (`libtvm4j_runtime_packed.so`) posicionada em [`android/mlc4j/output/arm64-v8a/`](demo3_android_mlc/mlc-llm-src/android/mlc4j/output/arm64-v8a/) e o catálogo do modelo em [`assets/mlc-app-config.json`](demo3_android_mlc/mlc-llm-src/android/MLCChat/app/src/main/assets/mlc-app-config.json), compilamos o aplicativo Android nativo (**MLCChat**, construído em Kotlin + Jetpack Compose):

- **Por que esta célula funciona em qualquer PC Windows com Android Studio?**
  1. Detecta automaticamente o `JAVA_HOME` apontando para o **JBR (*JetBrains Runtime*)** embutido no Android Studio (`C:\Program Files\Android\Android Studio\jbr`).
  2. Detecta automaticamente o caminho do `Android SDK` do usuário atual (`%LOCALAPPDATA%\Android\Sdk`) e grava o arquivo `local.properties` antes de invocar o Gradle.
  3. Executa o `gradlew.bat assembleDebug` e copia o instalador assinado para **`demo3_android_mlc/qwen2.5-mlc-chat.apk`**.

> 🔧 **Detalhes importantes de Engenharia Android aplicados no projeto [`MLCChat`](demo3_android_mlc/mlc-llm-src/android/MLCChat):**
> - **Compatibilidade JNI (`org.apache.tvm` v0.17.0):** As classes Java do módulo `:mlc4j` foram alinhadas com os símbolos JNI (`Java_org_apache_tvm_LibInfo_tvmFuncGetGlobal`) exportados pela `libtvm4j_runtime_packed.so`.
> - **Armazenamento Interno Privado (`application.filesDir`):** No [`AppViewModel.kt`](demo3_android_mlc/mlc-llm-src/android/MLCChat/app/src/main/java/ai/mlc/mlcchat/AppViewModel.kt#L47), configuramos `appDirFile = application.filesDir` (`/data/user/0/ai.mlc.mlcchat/files`) em vez de `getExternalFilesDir("")`, evitando bloqueios de permissão (`EACCES`) do *Scoped Storage* (FUSE) no Android 14/15.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

ANDROID_PROJECT_DIR = Path("./demo3_android_mlc/mlc-llm-src/android/MLCChat")
FINAL_APK_PATH = Path("./demo3_android_mlc/qwen2.5-mlc-chat.apk")

# 1. Detecta automaticamente o JAVA_HOME do Android Studio (JBR - JetBrains Runtime) caso não esteja no PATH
env = os.environ.copy()
jbr_default = Path(r"C:\Program Files\Android\Android Studio\jbr")
if jbr_default.exists() and "JAVA_HOME" not in env:
    env["JAVA_HOME"] = str(jbr_default)
    print(f"☕ JAVA_HOME configurado automaticamente para: {jbr_default}")

# 2. Atualiza o local.properties dinamicamente para o Android SDK do usuário atual (portável entre máquinas!)
local_sdk = Path(os.environ.get("LOCALAPPDATA", "")) / "Android" / "Sdk"
if local_sdk.exists():
    # O formato do local.properties do Gradle exige barras invertidas e dois-pontos escapados (C\:\Users\...)
    sdk_escaped = str(local_sdk).replace("\\", "\\\\").replace(":", "\\:")
    (ANDROID_PROJECT_DIR / "local.properties").write_text(f"sdk.dir={sdk_escaped}\n", encoding="utf-8")
    print(f"🔧 Android SDK detectado automaticamente em: {local_sdk}")

# 3. Garante que a biblioteca JNI nativa (libtvm4j_runtime_packed.so, ~98 MB) esteja presente em mlc4j/output/arm64-v8a
# (Em um git clone novo, caso o binário de 98 MB não esteja no git, extrai automaticamente da release oficial do MLC LLM)
jni_so_path = Path("./demo3_android_mlc/mlc-llm-src/android/mlc4j/output/arm64-v8a/libtvm4j_runtime_packed.so")
if not jni_so_path.exists():
    import urllib.request
    import zipfile
    base_apk = Path("./demo3_android_mlc/mlc-chat-base.apk")
    if not base_apk.exists():
        print("📥 Baixando runtime pré-compilado Android (libtvm4j_runtime_packed.so) da release oficial...")
        urllib.request.urlretrieve(
            "https://github.com/mlc-ai/binary-mlc-llm-libs/releases/download/Android-09262024/mlc-chat.apk",
            str(base_apk),
        )
    jni_so_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(base_apk, "r") as zf:
        jni_so_path.write_bytes(zf.read("lib/arm64-v8a/libtvm4j_runtime_packed.so"))
    print(f"✅ Biblioteca JNI extraída em: {jni_so_path}")

# 4. Invoca o Gradle Wrapper (gradlew.bat assembleDebug) para compilar o código Kotlin/Compose e empacotar o APK
print("🏗️ Executando Gradle (assembleDebug) para gerar o APK Android...")
gradle_cmd = [str((ANDROID_PROJECT_DIR / "gradlew.bat").resolve()), "assembleDebug"]
res = subprocess.run(
    gradle_cmd,
    cwd=str(ANDROID_PROJECT_DIR),
    env=env,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

# 5. Copia o APK gerado em app/build/outputs/apk/debug/app-debug.apk para demo3_android_mlc/qwen2.5-mlc-chat.apk
if res.returncode == 0:
    built_apk = ANDROID_PROJECT_DIR / "app" / "build" / "outputs" / "apk" / "debug" / "app-debug.apk"
    shutil.copy2(built_apk, FINAL_APK_PATH)
    apk_mb = FINAL_APK_PATH.stat().st_size / (1024 ** 2)
    print(f"✅ BUILD SUCCESSFUL! APK gerado em: {FINAL_APK_PATH.resolve()} ({apk_mb:.1f} MB)")
else:
    print("⚠️ Log do Gradle:")
    print((res.stdout + "\n" + res.stderr)[-1500:])
    if FINAL_APK_PATH.exists():
        print(f"✅ APK pré-compilado já disponível em: {FINAL_APK_PATH.resolve()}")

### Passo 3.4 — Instalação no Celular via USB ([`adb`](https://developer.android.com/tools/adb)) + Pesos 100% Offline

Para que a demonstração ao vivo na sala de aula seja **100% à prova de falhas de internet/Wi-Fi** e compatível com as regras de segurança do **Android 14/15**:

1. Conecte seu smartphone Android via cabo USB com a **Depuração USB** ativada nas *Opções do Desenvolvedor*.
2. A célula abaixo localiza automaticamente o executável `adb.exe` e executa 3 passos:
   - **`adb install -r qwen2.5-mlc-chat.apk`:** Instala (ou atualiza mantendo os dados com `-r`) o aplicativo no telefone.
   - **Provisionamento Offline com Permissões Nativas (`run-as ai.mlc.mlcchat`):**
     - No Android 14+, arquivos enviados diretamente via `adb push` pertencem ao usuário `shell` (`uid=2000`), o que causa erro `EACCES (Permission denied)` quando o app tenta abri-los no *Scoped Storage*.
     - Para resolver isso de forma limpa, fazemos `adb push` para a pasta temporária pública `/data/local/tmp/` e usamos `adb shell run-as ai.mlc.mlcchat cp -r ... files/`, fazendo com que **o próprio usuário Linux do aplicativo (`u0_a500`) crie e seja dono dos arquivos** dentro de `/data/user/0/ai.mlc.mlcchat/files/`!
   - **`adb shell am start -n ai.mlc.mlcchat/.MainActivity`:** Abre o aplicativo automaticamente na tela do seu celular, pronto para conversar **mesmo em Modo Avião**!

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path
from huggingface_hub import snapshot_download

FINAL_APK_PATH = Path("./demo3_android_mlc/qwen2.5-mlc-chat.apk")
OFFLINE_WEIGHTS_DIR = Path("./models/qwen2.5-1.5b-mlc")
MODEL_ID = "Qwen2.5-1.5B-Instruct-q4f16_1-MLC"

# 1. Localiza o executável adb (primeiro verifica no PATH do sistema; se não achar, busca no Android SDK padrão)
adb_exe = shutil.which("adb")
if not adb_exe:
    sdk_adb = Path(os.environ.get("LOCALAPPDATA", "")) / "Android" / "Sdk" / "platform-tools" / "adb.exe"
    if sdk_adb.exists():
        adb_exe = str(sdk_adb)

if not adb_exe:
    print("❌ 'adb' não encontrado. Verifique se o Android SDK Platform-Tools está instalado.")
else:
    print(f"🔌 Usando ADB: {adb_exe}")
    # Lista os dispositivos USB conectados ao servidor ADB
    devs = subprocess.run([adb_exe, "devices"], capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(devs.stdout.strip())

    # Filtra apenas linhas que terminam com '\tdevice' (ignorando linhas vazias ou aparelhos 'unauthorized')
    lines = [l.strip() for l in devs.stdout.strip().splitlines()[1:] if l.strip()]
    authorized = [l for l in lines if "\tdevice" in l]

    if not authorized:
        print("\n📱 Nenhum celular detectado no momento (ou aguardando autorização USB na tela do aparelho).")
        print(f"👉 O APK já está pronto em: {FINAL_APK_PATH.resolve()}")
        print("   Conecte o celular com 'Depuração USB' ativada e rode esta célula novamente para instalar!")
    else:
        device_serial = authorized[0].split()[0]
        print(f"\n📲 Dispositivo Android conectado: {device_serial}")

        # Passo A: Instala/atualiza (-r = replace existing application) o APK no smartphone
        print("1️⃣ Instalando o APK (qwen2.5-mlc-chat.apk) no smartphone...")
        inst = subprocess.run(
            [adb_exe, "install", "-r", str(FINAL_APK_PATH)],
            capture_output=True, text=True, encoding="utf-8", errors="replace"
        )
        print(inst.stdout or inst.stderr)

        # Passo B: Verifica se os 30 shards de pesos já estão presentes no armazenamento interno privado do app
        chk = subprocess.run(
            [adb_exe, "shell", f"run-as ai.mlc.mlcchat ls files/{MODEL_ID}/params_shard_29.bin"],
            capture_output=True, text=True, encoding="utf-8", errors="replace"
        )
        if chk.returncode == 0:
            print(f"✅ Pesos {MODEL_ID} já estão provisionados no armazenamento interno do app (100% Offline)!")
        else:
            # Garante que os shards estejam baixados no computador antes de enviar via USB
            if len(list(OFFLINE_WEIGHTS_DIR.glob("params_shard_*.bin"))) < 30:
                print("📥 Baixando pesos q4f16_1 para provisionamento offline...")
                snapshot_download(repo_id=f"mlc-ai/{MODEL_ID}", local_dir=str(OFFLINE_WEIGHTS_DIR))

            # Envia para /data/local/tmp e copia como o próprio usuário do app via 'run-as ai.mlc.mlcchat'
            # Isso garante que os arquivos pertençam ao UID do app (evitando erro EACCES no Android 14/15)
            tmp_dir = f"/data/local/tmp/{MODEL_ID}"
            print(f"2️⃣ Enviando pesos quantizados via USB para {tmp_dir} e aplicando permissões do app...")
            subprocess.run([adb_exe, "shell", "mkdir", "-p", tmp_dir], check=False)
            push = subprocess.run(
                [adb_exe, "push", f"{OFFLINE_WEIGHTS_DIR.resolve()}/.", tmp_dir],
                capture_output=True, text=True, encoding="utf-8", errors="replace"
            )
            print((push.stdout or push.stderr)[-400:])
            subprocess.run([
                adb_exe, "shell",
                f"chmod -R 755 {tmp_dir} && run-as ai.mlc.mlcchat mkdir -p files && run-as ai.mlc.mlcchat cp -r {tmp_dir} files/ && rm -rf {tmp_dir}"
            ], check=False)

        # Passo C: Reinicia a Activity principal para abrir o app imediatamente na tela do celular
        subprocess.run([adb_exe, "shell", "am", "force-stop", "ai.mlc.mlcchat"], check=False)
        subprocess.run([adb_exe, "shell", "am", "start", "-n", "ai.mlc.mlcchat/.MainActivity"], check=False)
        print("\n🎉 Tudo pronto! O app 'MLCChat' já foi aberto no seu celular com o modelo pronto para chat 100% offline!")

---
### 🎤 Nota para Apresentação Oral: Alternativa com ONNX Runtime Mobile (`onnxruntime-android`)

Durante a apresentação, vale destacar uma excelente comparação arquitetural para a banca/audiência:

> *"Se na **Demo 1** nós já exportamos o modelo para `.onnx`, por que usamos **MLC LLM** na **Demo 3** em vez de rodar o próprio ONNX no Android?"*

1. **Quando usar [MLC LLM / Apache TVM](https://llm.mlc.ai/docs/deploy/android.html):**
   - É o estado da arte em **velocidade de decodificação de LLMs em GPUs mobile** (Adreno/Mali via OpenCL/Vulkan), pois compila *kernels* específicos fundindo a desquantização `int4 -> fp16` com a multiplicação matricial.
2. **Quando usar [ONNX Runtime Mobile](https://onnxruntime.ai/docs/get-started/with-java.html#android) (`com.microsoft.onnxruntime:onnxruntime-android` + [`onnxruntime-genai`](https://onnxruntime.ai/docs/genai/)):**
   - Quando a equipe de engenharia quer **padronizar um único artefato (`.onnx`)** do Cloud até o Mobile.
   - Com o **ONNX Runtime GenAI**, é possível quantizar o `.onnx` em INT4 (`matmul_4bits_quantizer`), adicionar a dependência Maven `onnxruntime-android` no `build.gradle` e delegar a execução para a [**NNAPI**](https://onnxruntime.ai/docs/execution-providers/NNAPI-ExecutionProvider.html) do Android ou [**Qualcomm QNN (NPU Hexagon)**](https://onnxruntime.ai/docs/execution-providers/QNN-ExecutionProvider.html) sem precisar compilar bibliotecas `.tar`/`.so` específicas por modelo!

---
## 🏁 Conclusão Geral: Do Cloud ao Edge Mobile

| Dimensão | Demo 1: Cloud / Microsserviço | Demo 2: Edge Local / Desktop | Demo 3: Mobile Nativo (Android) |
| :--- | :--- | :--- | :--- |
| **Motor de Inferência** | **ONNX Runtime** (`CPUExecutionProvider`) | **llama.cpp / Ollama** + Open WebUI | **MLC LLM (Apache TVM)** + OpenCL |
| **Formato & Precisão** | `.onnx` + `.onnx_data` (`FP32` — 32 bits) | `.gguf` (`Q8_0` — 8 bits) | `params_shard_*.bin` (`q4f16_1` — 4 bits) |
| **Tamanho em Disco** | **~1890.4 MB** | **~506.5 MB** (`-73%`) | **~265.1 MB** (`-86%`) |
| **Estratégia de Memória** | Grafo otimizado em RAM + KV-Cache explícito | *Memory-Mapped File* (`mmap`) direto do SSD | Quantização em registrador GPU + Chunked Prefill (`628 MB` RAM) |
| **Interface de Uso** | REST API FastAPI + Swagger UI (`:8000`) | API `:11435` + Chat WebUI (`:3000`) | App Nativo Android (`.apk` 100% Offline) |
| **Dependência de PyTorch em Produção** | **Zero** (Apenas `onnxruntime` + `numpy`) | **Zero** (Binário C/C++ `llama.cpp`) | **Zero** (Runtime C++/JNI `tvm4j`) |